# How an agent works

You have probably heard about **AI agents**: tools like Claude Code or GitHub Copilot that can look things up, run code and take several steps on their own before they answer.

This notebook builds a small agent from scratch, so you can see every part of it. The main idea is simple:

> An agent is a language model inside a **loop**. The model asks for a tool, ordinary Python code runs the tool, the result goes back to the model, and the loop repeats until the model gives an answer.

Our agent will answer questions about **Fantasy Premier League** players by writing SQL queries against a small database. Everything runs in your browser.

The model we use is small, so it will make mistakes. That is useful: each mistake shows you a part of the agent that real products have to handle too.

## Step 1: Get set up

Small cells that bring in the Python tools we need, check that your browser can run models, load the WebLLM library, and set up the model loader. You have seen this setup in the earlier notebooks.

### 1a: Import tools

- `json` turns Python data into text and back. We use it to hand data to JavaScript.
- `ast` reads a piece of Python code without running it (we use it for a safe calculator).
- `csv` reads spreadsheet-style text files.
- `operator` gives plain functions for `+`, `-`, `*` and `/`.
- `re` finds patterns in text (these patterns are called **regular expressions**).
- `sqlite3` creates and searches a small **database** (an organized store of data you can search).
- `pyjs` connects Python to the browser's **JavaScript**, the language web pages run on. WebLLM is written in JavaScript, so we need this bridge.
- `display` lets a cell show something and then update it in place (we use it for a progress bar).

In [ ]:
import json
import ast
import csv
import operator
import re
import sqlite3
import pyjs
from IPython.display import display

js = pyjs.js  # a doorway from Python into the browser's JavaScript

print("Tools imported.")

### 1b: Check for WebGPU

Models run on your computer's graphics chip (GPU). Browsers reach the GPU through a feature called **WebGPU**. Recent Chrome and Edge on a laptop have it; many phones and older browsers do not.

In [ ]:
if getattr(js.navigator, "gpu", None) is not None:
    print("WebGPU is available. You can run models.")
else:
    print("WebGPU is not available here. Try the latest Chrome or Edge on a laptop.")

### 1c: Load WebLLM

**WebLLM** is a free JavaScript library that runs language models inside the browser. This line fetches it from the internet (it takes a few seconds the first time).

In [ ]:
webllm = await js.eval("import('https://esm.run/@mlc-ai/web-llm')")
print("WebLLM loaded.")

### 1d: Translate between Python and JavaScript

WebLLM runs in JavaScript, and JavaScript cannot read Python dictionaries directly. Both languages understand **JSON**, a plain-text format for data, so we translate by way of JSON text:

- `js_obj()` turns a Python dictionary into a JavaScript object, for requests we send to WebLLM.
- `py_obj()` goes the other way, turning a JavaScript object (like the model's reply) into Python dictionaries and lists.

You met `js_obj()` in Step 7c of the starter notebook.

In [ ]:
def js_obj(d):
    """Turn a Python dict into a JavaScript object (by way of JSON text)."""
    return js.JSON.parse(json.dumps(d))


def py_obj(value):
    """Turn a JavaScript object into Python dicts and lists (by way of JSON text)."""
    return json.loads(js.JSON.stringify(value))


print("Translators ready.")

### 1e: Set up the model loader

- `engine` is the part of WebLLM that actually runs a model. It starts as `None` (Python for "nothing yet") and gets created the first time we need it.
- `loaded_model` remembers the name of the model that is currently loaded, so later steps can check that one is ready.
- `progress_bar()` shows a bar that fills up while a model downloads or loads.
- `start_model()` gets a model ready to run. If the model is not saved in your browser yet, it downloads it first.

This is the same loader as in the starter notebook. You do not need to follow every line. Just run it.

In [ ]:
engine = None        # the thing that runs a model
loaded_model = None  # which model is currently loaded


def progress_bar():
    """Show a progress bar that updates in place."""
    status = display("Starting...", display_id=True)

    def show(report):
        filled = int(report.progress * 30)
        bar = "#" * filled + "-" * (30 - filled)
        status.update(f"[{bar}] {report.progress:.0%}\n{report.text}")

    # JavaScript needs its own handle on the Python function to call it
    js_show, handle = pyjs.create_callable(show)
    return status, js_show, handle


async def start_model(model_id):
    """Get a model ready to run. Downloads it only if it is not already saved."""
    global engine
    status, progress, handle = progress_bar()

    if engine is None:
        engine = webllm.MLCEngine.new()   # create an empty engine
    engine.setInitProgressCallback(progress)

    try:
        await engine.reload(model_id)
    except Exception as error:
        # Some models have two conflicting memory-window settings.
        # If so, turn off the "sliding window" one and try again.
        if "WindowSizeConfigurationError" in str(error):
            status.update("Adjusting this model's settings and retrying...")
            await engine.reload(model_id, js_obj({"sliding_window_size": -1}))
        else:
            raise

print("Model loader is ready.")

## Step 2: Choose a model

Agents need a model that can follow a strict output format. The smallest models usually cannot, so this notebook uses slightly larger ones:

| Model | Download | Why |
|---|---|---|
| `Qwen3-1.7B-q4f16_1-MLC` | about 2.0 GB | A good start on most laptops. It makes more mistakes than Qwen3 4B |
| `Qwen3-4B-q4f16_1-MLC` | about 3.4 GB | The best of the three at following formats and using tools, but slower and needs a stronger laptop |
| `Hermes-3-Llama-3.2-3B-q4f16_1-MLC` | about 2.3 GB | Trained specially for tool use, from a different model family |

Start with Qwen3 1.7B. If it runs well on your laptop, try Qwen3 4B. Later, in Step 13, you can come back and compare all three.

**A note on Qwen3:** Qwen3 models usually "think out loud" in a `<think>` block before answering, which is slow. Adding `/no_think` to the instructions turns this off, and this notebook does that for you.

In [ ]:
MODEL_NAME = "Qwen3-1.7B-q4f16_1-MLC"

print(f"Model to load: {MODEL_NAME}")
if await webllm.hasModelInCache(MODEL_NAME):
    print("This model is already saved in your browser.")
else:
    print("This model is not saved yet, so the next step will download it.")

## Step 3: Load the model

The first download can take several minutes. After that, the model is saved in your browser.

In [ ]:
await start_model(MODEL_NAME)
loaded_model = MODEL_NAME
print(f"{loaded_model} is loaded and ready.")

## Step 4: Load the data

The data is one table of Premier League players from the 2025-26 Fantasy Premier League season, after gameweek 29. Each row is one player.

| Column | Meaning |
|---|---|
| `name` | Short name, as used in the game (for example `Haaland`) |
| `full_name` | First and last name |
| `team` | Club |
| `position` | Goalkeeper, Defender, Midfielder or Forward |
| `price` | Price in the game, in millions of pounds |
| `total_points` | Fantasy points so far this season |
| `goals`, `assists`, `minutes` | Season totals |
| `form` | Average points over recent games |
| `selected_by_percent` | Share of Fantasy teams that include the player |

We load the file into a small **SQLite** database that lives in memory while this notebook is open.

In [ ]:
DATA_PATH = "data/fpl_players.csv"

db = sqlite3.connect(":memory:")
db.execute("""
CREATE TABLE players (
    name TEXT, full_name TEXT, team TEXT, position TEXT,
    price REAL, total_points INTEGER, goals INTEGER, assists INTEGER,
    minutes INTEGER, form REAL, selected_by_percent REAL
)
""")

with open(DATA_PATH, newline="", encoding="utf-8") as f:
    rows = [
        (r["name"], r["full_name"], r["team"], r["position"],
         float(r["price"]), int(r["total_points"]), int(r["goals"]), int(r["assists"]),
         int(r["minutes"]), float(r["form"]), float(r["selected_by_percent"]))
        for r in csv.DictReader(f)
    ]

db.executemany("INSERT INTO players VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)", rows)
TEAMS = [t for (t,) in db.execute("SELECT DISTINCT team FROM players ORDER BY team")]
print(f"Loaded {len(rows)} players from {len(TEAMS)} teams.")

## Step 5: Build the tools

A **tool** is just an ordinary Python function. The model cannot run code itself. It can only *ask* for a tool, and our code decides whether and how to run it.

Our agent gets two tools:

- `run_sql` runs a SQL query on the players table and returns the result as text.
- `calculator` does arithmetic, because language models are often bad at it.

### 5a: The SQL tool

Two rules are built into this tool, and both are part of making an agent safe and workable:

- It only allows `SELECT` queries, so the model cannot change or delete the data.
- It returns at most `MAX_ROWS` rows. The model can only read a limited amount of text at once (its **context window**, about 4,000 tokens here), so a query that returned all 819 players would not fit.

In [ ]:
MAX_ROWS = 10


def run_sql(query):
    """Run one SELECT query and return the result as a short text table."""
    if not query.strip().lower().startswith(("select", "with")):
        return "Error: only SELECT queries are allowed."
    try:
        cursor = db.execute(query)
    except Exception as error:
        return f"SQL error: {error}"
    columns = [d[0] for d in cursor.description]
    rows = cursor.fetchmany(MAX_ROWS + 1)
    if not rows:
        return "The query returned no rows."
    lines = [" | ".join(columns)]
    for row in rows[:MAX_ROWS]:
        lines.append(" | ".join(str(value) for value in row))
    if len(rows) > MAX_ROWS:
        lines.append(f"(only the first {MAX_ROWS} rows are shown)")
    return "\n".join(lines)

Try the tool yourself before giving it to the model. Change the query and run it again.

In [ ]:
QUERY = "SELECT name, team, total_points FROM players ORDER BY total_points DESC LIMIT 5"

print(run_sql(QUERY))

### 5b: The calculator tool

The calculator only accepts numbers and the operators `+ - * / ( )`. It reads the expression with Python's `ast` module instead of `eval`, so it cannot run any other code.

In [ ]:
OPERATORS = {
    ast.Add: operator.add, ast.Sub: operator.sub,
    ast.Mult: operator.mul, ast.Div: operator.truediv,
    ast.USub: operator.neg,
}


def evaluate(node):
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in OPERATORS:
        return OPERATORS[type(node.op)](evaluate(node.left), evaluate(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in OPERATORS:
        return OPERATORS[type(node.op)](evaluate(node.operand))
    raise ValueError("not allowed")


def calculator(expression):
    """Do simple arithmetic and return the result as text."""
    try:
        value = evaluate(ast.parse(expression, mode="eval").body)
    except Exception:
        return "Calculator error: use only numbers and + - * / ( )"
    return str(round(value, 4))


print(calculator("193 / 14.7"))
print(calculator("import os"))

### 5c: Put the tools in a dictionary

The agent loop will look up tools by name in this dictionary.

In [ ]:
TOOLS = {
    "run_sql": run_sql,
    "calculator": calculator,
}

print("Tools available:", list(TOOLS))

## Step 6: Tell the model about the tools

The model only knows what we put in its messages. The **system message** below describes the table, the tools, and the exact format the model must use to ask for a tool. Read it: this text is most of what makes the agent work.

We ask the model to reply with a small piece of **JSON**, a text format for structured data, with two fields:

- `action`: which tool to use, or `final_answer`
- `input`: the SQL query, the arithmetic, or the answer

In [ ]:
SYSTEM_PROMPT = f"""You answer questions about Fantasy Premier League players by using tools.

There is one SQLite table called players with these columns:
name (short name, for example Haaland), full_name, team, position, price (millions of pounds),
total_points, goals, assists, minutes, form, selected_by_percent.
position is one of: Goalkeeper, Defender, Midfielder, Forward.
team is one of: {", ".join(TEAMS)}.

Tools:
- run_sql: runs one SELECT query on the players table and returns up to {MAX_ROWS} rows.
- calculator: does arithmetic, for example 193 / 14.7

Reply with exactly one JSON object and nothing else, in one of these forms:
{{"action": "run_sql", "input": "SELECT ..."}}
{{"action": "calculator", "input": "193 / 14.7"}}
{{"action": "final_answer", "input": "a one-sentence answer"}}

Always use a tool to look up facts. Give the final answer only after you have seen a tool result."""


def system_message():
    """The system message, with /no_think added for Qwen3 models."""
    text = SYSTEM_PROMPT
    if loaded_model and "Qwen3" in loaded_model:
        text += " /no_think"
    return {"role": "system", "content": text}


print(system_message()["content"])

### Before you go on: check that a model is loaded

Steps 7 to 13 need the model from Step 3. This cell checks.

In [ ]:
if loaded_model is None:
    print("No model is loaded yet. Go back to Step 3 and run it until it says the model is ready.")
else:
    print(f"Ready to use {loaded_model}.")

## Step 7: One step by hand

Before building a loop, we do one step ourselves so you can see exactly what happens.

### 7a: Build the messages and the request

In [ ]:
QUESTION = "Which player has the most total points?"

messages = [
    system_message(),
    {"role": "user", "content": QUESTION},
]

request_dict = {
    "messages": messages,
    "temperature": 0.0,
    "max_tokens": 300,
    "frequency_penalty": 0.3,
}
print(f"Sending {len(messages)} messages.")

### 7b: Send it and look at the raw reply

The model's reply is plain text. If everything goes well, the text is a JSON object asking for a tool.

In [ ]:
reply = await engine.chat.completions.create(js_obj(request_dict))
raw_reply = py_obj(reply)["choices"][0]["message"]["content"]
print(raw_reply)

### 7c: Read the JSON

The next helper finds the JSON object in the reply and checks that it has the two fields we asked for. If the model wrote something else, it raises an error that explains the problem. It also removes any `<think>` block first, in case the model thought out loud anyway.

In [ ]:
def parse_action(text):
    """Find the JSON object in the model's reply. Return (action, input)."""
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()
    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if match is None:
        raise ValueError("The reply did not contain a JSON object.")
    try:
        data = json.loads(match.group(0))
    except json.JSONDecodeError as error:
        raise ValueError(f"The reply was not valid JSON ({error}).")
    if "action" not in data or "input" not in data:
        raise ValueError('The JSON must have an "action" and an "input".')
    return str(data["action"]), str(data["input"])

Now try it on the reply from 7b. If the model did not follow the format, you will see a friendly message instead of an error.

In [ ]:
try:
    action, action_input = parse_action(raw_reply)
    print("action:", action)
    print("input: ", action_input)
except ValueError as error:
    action = None
    print("The model did not follow the format:", error)
    print("Run Step 7b again, or try another model in Step 2.")

## Step 8: Run the tool and send the result back

Now **our code** runs the tool the model asked for. Then we add two messages to the conversation: the model's own request, and the tool result.

We send the tool result as a `user` message that starts with "Tool result". Some systems use a special `tool` role for this, but the idea is the same: the result is just more text in the conversation.

In [ ]:
if action not in TOOLS:
    print(f"The model asked for {action!r}, not a tool. Run Step 7b again to get a tool request.")
else:
    tool_result = TOOLS[action](action_input)
    print("Tool result:")
    print(tool_result)

    messages.append({"role": "assistant", "content": raw_reply})
    messages.append({"role": "user", "content": "Tool result:\n" + tool_result})
    print()
    print(f"The conversation now has {len(messages)} messages.")

Send the longer conversation back to the model. This time it should give a final answer.

In [ ]:
request_dict["messages"] = messages
reply = await engine.chat.completions.create(js_obj(request_dict))
raw_reply = py_obj(reply)["choices"][0]["message"]["content"]
print(raw_reply)

That was a complete agent step: **model asks, code acts, result goes back**. Everything else in this notebook repeats this pattern and handles what can go wrong.

## Step 9: Put it in a loop

### 9a: A helper for asking the model

This wraps the request from Step 7 so we can call it many times. The `json_mode` setting is used later, in Step 11.

In [ ]:
ACTION_SCHEMA = json.dumps({
    "type": "object",
    "properties": {
        "action": {"type": "string", "enum": ["run_sql", "calculator", "final_answer"]},
        "input": {"type": "string"},
    },
    "required": ["action", "input"],
})


async def ask(messages, json_mode=False):
    """Send the messages to the model and return the text of its reply."""
    request = {
        "messages": messages,
        "temperature": 0.0,
        "max_tokens": 300,
        "frequency_penalty": 0.3,
    }
    if json_mode:
        request["response_format"] = {"type": "json_object", "schema": ACTION_SCHEMA}
    reply = await engine.chat.completions.create(js_obj(request))
    return py_obj(reply)["choices"][0]["message"]["content"]

### 9b: The simplest possible agent loop

This is the whole idea in a few lines. It has no protection against mistakes: if the model writes bad JSON or asks for a tool that does not exist, the loop crashes.

In [ ]:
async def run_agent_simple(question):
    messages = [system_message(), {"role": "user", "content": question}]
    while True:
        raw = await ask(messages)
        print("MODEL:", raw)
        action, action_input = parse_action(raw)
        if action == "final_answer":
            return action_input
        result = TOOLS[action](action_input)
        print("TOOL: ", result)
        print()
        messages.append({"role": "assistant", "content": raw})
        messages.append({"role": "user", "content": "Tool result:\n" + result})

Try the simple loop on one question. If the model breaks the format, this cell will stop with an error; Step 10 fixes that.

In [ ]:
QUESTION = "Which defender has the most total points?"

answer = await run_agent_simple(QUESTION)
print()
print("ANSWER:", answer)

## Step 10: Make the loop sturdier

Real agents spend much of their code on what can go wrong. With a small model you will see these problems often:

- **Bad format:** the reply is not valid JSON, or is missing a field.
- **Unknown tool:** the model asks for a tool that does not exist.
- **Tool errors:** the SQL has a mistake, such as a misspelled column.
- **Loops:** the model keeps asking for tools and never answers.

The improved loop below handles each one. Instead of crashing, it tells the model what went wrong and lets it try again. It also stops after `MAX_STEPS` steps, so it cannot run forever or overflow the context window.

The rules for these problems are often called the **harness**: the code around the model that turns it into a working system.

In [ ]:
MAX_STEPS = 5


async def run_agent(question, json_mode=False, verbose=True):
    """Run the agent loop. Return a dict with the answer and what happened."""
    messages = [system_message(), {"role": "user", "content": question}]
    tool_calls = 0
    problems = 0
    for step in range(1, MAX_STEPS + 1):
        raw = await ask(messages, json_mode=json_mode)
        messages.append({"role": "assistant", "content": raw})
        if verbose:
            print(f"[step {step}] MODEL: {raw.strip()}")

        try:
            action, action_input = parse_action(raw)
        except ValueError as error:
            problems += 1
            feedback = f"{error} Reply with exactly one JSON object."
        else:
            if action == "final_answer":
                return {"answer": action_input, "steps": step,
                        "tool_calls": tool_calls, "problems": problems}
            if action not in TOOLS:
                problems += 1
                feedback = f"There is no tool called {action}. Use run_sql, calculator or final_answer."
            else:
                tool_calls += 1
                feedback = "Tool result:\n" + TOOLS[action](action_input)

        messages.append({"role": "user", "content": feedback})
        if verbose:
            print(f"[step {step}] REPLY TO MODEL: {feedback}")
            print()

    return {"answer": None, "steps": MAX_STEPS, "tool_calls": tool_calls, "problems": problems}

Try it on a question that needs a filter, and look at every step it prints.

In [ ]:
QUESTION = "Which midfielder with a price under 6.0 has the most total points?"

result = await run_agent(QUESTION)
print()
print("ANSWER:", result["answer"])
print(f"Steps: {result['steps']}, tool calls: {result['tool_calls']}, problems: {result['problems']}")

### 10a: Questions that need two tools

Some questions need a query and then some arithmetic. Watch whether the model uses the calculator or tries to do the arithmetic itself.

In [ ]:
QUESTION = "How many points per million did Haaland score? Divide his total points by his price."

result = await run_agent(QUESTION)
print()
print("ANSWER:", result["answer"])

## Step 11: Force the format with JSON mode

WebLLM can **force** the model's reply to match a JSON schema. A **schema** describes what the JSON must look like: here, an `action` that must be one of our three names, and an `input` that must be text. While the model writes, WebLLM blocks any token that would break the schema.

This removes most format problems, but not all: the JSON will be valid, but the SQL inside it can still be wrong.

Run the same question with and without JSON mode and compare the `problems` count.

In [ ]:
print(ACTION_SCHEMA)

Now run one question twice, first without and then with JSON mode.

In [ ]:
QUESTION = "Which team's players have scored the most goals in total?"

for mode in [False, True]:
    print(f"===== json_mode={mode} =====")
    result = await run_agent(QUESTION, json_mode=mode)
    print("ANSWER:", result["answer"], "| problems:", result["problems"])
    print()

## Step 12: Measure how well the agent works

Watching a few examples is not enough to know whether an agent works. We need to **evaluate** it: run it on questions with known answers and count how many it gets right.

### 12a: A small test set

Each question comes with the answer text we expect to find in the agent's final answer. The answers were found by running SQL ourselves.

In [ ]:
EVAL_SET = [
    ("Which player has the most total points?", "Haaland"),
    ("Which player has the most assists?", "Fernandes"),
    ("Which team's players have scored the most goals in total?", "Man City"),
    ("Who is the most expensive goalkeeper?", "Raya"),
    ("Which midfielder with a price under 6.0 has the most total points?", "Garner"),
    ("How many Arsenal defenders are in the table?", "13"),
    ("What is Haaland's total points divided by his price, to one decimal place?", "13.1"),
    ("Which defender has the most total points?", "Gabriel"),
]


def is_correct(answer, expected):
    return answer is not None and expected.lower() in answer.lower()


print(f"{len(EVAL_SET)} test questions.")

### 12b: Run the test

This runs the agent on every question without printing each step. It can take a few minutes.

In [ ]:
USE_JSON_MODE = True

score = 0
total_problems = 0
for question, expected in EVAL_SET:
    result = await run_agent(question, json_mode=USE_JSON_MODE, verbose=False)
    correct = is_correct(result["answer"], expected)
    score += correct
    total_problems += result["problems"]
    mark = "right" if correct else "WRONG"
    print(f"{mark:5} | {question}")
    print(f"      | expected {expected!r}, got {result['answer']!r}")
print()
print(f"{loaded_model}: {score} of {len(EVAL_SET)} correct, {total_problems} format problems")

## Step 13: Compare models

Change `MODEL_NAME` in Step 2, run Steps 2 and 3 to load the other model, and run Step 12b again. Record each score here so you can compare them.

Questions to think about:

- Does the larger model score better? By how much?
- Does Hermes 3, which was trained for tool use, do better than a Qwen3 model of similar size?
- How much does JSON mode help each model?

In [ ]:
SCORES = {
    # "Qwen3-4B, JSON mode": 0,
    # "Qwen3-1.7B, JSON mode": 0,
    # "Hermes-3-Llama-3.2-3B, JSON mode": 0,
}

for name, value in SCORES.items():
    print(f"{name}: {value} of {len(EVAL_SET)}")

## Step 14: From this notebook to real agents

Tools like Claude Code, GitHub Copilot and ChatGPT's agents use the same loop you built here. The differences are of scale, not of kind:

- **A stronger model** that makes fewer format mistakes and plans better.
- **More tools**, such as reading and editing files, running programs, and searching the web.
- **A much larger harness**: permissions for risky actions, longer context windows, ways to summarize old steps, and extensive evaluation.

The core is still the same: the model asks, ordinary code acts, and the result goes back into the conversation.

## Your turn

- Add a question to `EVAL_SET` and find its answer yourself with `run_sql` first.
- Write a third tool, for example one that returns the list of players on a given team, and describe it in the system message.
- Change the system message and see whether the score in Step 12 goes up or down.
- Set `MAX_ROWS` to 3 and see which questions start to fail.